# 19 — Intermediate Review and Readiness Assessment

Bring together classes, typing, iterators, testing, persistence, and application boundaries. This review provides a compact integration example and prepares you for the three portfolio projects.

*Intermediate · 75–120 min*

## You will learn

- Combine typed records and object-oriented design
- Separate storage, parsing, and business logic
- Test domain behavior without external I/O
- Identify topics to revisit before Advanced Python

## Examples

### 1. Dataclass domain model and validation

A small typed record is easy to inspect and test. Domain validation must explicitly enforce valid values, because annotations alone do not. A frozen record communicates an intent to avoid reassignment after creation.

In [ ]:
from dataclasses import dataclass
import math

@dataclass(frozen=True)
class Sale:
    sku: str
    quantity: int
    unit_price: float
    def __post_init__(self):
        if not self.sku.strip() or isinstance(self.quantity, bool) or not isinstance(self.quantity, int) or self.quantity <= 0:
            raise ValueError("invalid SKU or quantity")
        if not math.isfinite(self.unit_price) or self.unit_price < 0:
            raise ValueError("invalid price")
    @property
    def total(self):
        return self.quantity * self.unit_price

print(Sale("P1", 2, 6.5).total)

**Think about it:** What would happen if a floating-point quantity were passed despite the type annotation?

### 2. Separate parsing from analysis

Keep data conversion distinct from aggregation. An iterable of domain records can be streamed through a pure reducer, while each parser validates its own boundary. Avoid mutating source data during summarization.

In [ ]:
def parse_sale(raw):
    if not isinstance(raw, dict):
        raise ValueError("sale must be an object")
    return Sale(raw["sku"], int(raw["quantity"]), float(raw["unit_price"]))

def aggregate_sales(records):
    totals = {}
    for sale in records:
        totals[sale.sku] = totals.get(sale.sku, 0.0) + sale.total
    return totals

raw_rows = [{"sku": "P1", "quantity": "2", "unit_price": "6.5"},
            {"sku": "P2", "quantity": "3", "unit_price": "1.25"}]
print(aggregate_sales(map(parse_sale, raw_rows)))

**Think about it:** What extra validation would you add to avoid silently accepting float values as quantities?

### 3. Persistence without permanent side effects

Temporary SQLite storage is useful for demonstration and tests. Use parameterized SQL and explicitly close connections. Integration tests can verify that saved and loaded data match the original models.

In [ ]:
import sqlite3

db = sqlite3.connect(":memory:")
try:
    db.execute("CREATE TABLE sales (sku TEXT, quantity INTEGER, unit_price REAL)")
    db.executemany(
        "INSERT INTO sales VALUES (?, ?, ?)",
        [(sale.sku, sale.quantity, sale.unit_price) for sale in map(parse_sale, raw_rows)]
    )
    rows = db.execute("SELECT sku, SUM(quantity * unit_price) FROM sales GROUP BY sku ORDER BY sku").fetchall()
    print(rows)
finally:
    db.close()

**Think about it:** Why are parameter placeholders important even when a notebook uses sample data?

### 4. Review tests and key guarantees

Good tests verify behavior rather than incidental implementation details. Test normal cases, empty collections, malformed data, and boundary values. A code review should identify error policies, state ownership, and resource cleanup.

In [ ]:
import unittest

class SaleTests(unittest.TestCase):
    def test_total(self):
        self.assertEqual(Sale("A", 3, 4.0).total, 12)
    def test_bad_quantity(self):
        with self.assertRaises(ValueError):
            Sale("A", 0, 3.0)
    def test_aggregation(self):
        self.assertEqual(aggregate_sales([Sale("A", 1, 5.0), Sale("A", 2, 5.0)]), {"A": 15.0})

suite = unittest.defaultTestLoader.loadTestsFromTestCase(SaleTests)
assert unittest.TextTestRunner(verbosity=0).run(suite).wasSuccessful()

**Think about it:** What other edge cases might a real sales processing pipeline encounter?

## Exercises

**Exercise 1.** Design an application boundary between HTTP parsing and SQLite persistence.

In [ ]:
# Exercise 1
# Implement your solution here, and add tests.

**Exercise 2.** Write tests for duplicate records, invalid numeric inputs, and empty datasets.

In [ ]:
# Exercise 2
# Implement your solution here, and add tests.

**Exercise 3.** Explain composition vs inheritance with a practical example.

In [ ]:
# Exercise 3
# Implement your solution here, and add tests.

**Exercise 4.** Refactor a large notebook cell into functions with annotations and clear responsibility.

In [ ]:
# Exercise 4
# Implement your solution here, and add tests.

## Common mistakes

- Avoid state leaking between notebook cells.
- External data must be validated, even if a type annotation is present.
- Intermediate skill includes explanation, tests, and maintainability—not just syntax.

**Next:** the [intermediate projects](../projects/) and [exercise workbooks](../exercises/)